# Phase 3: PyTorch MLP — Flight Delay Prediction (GPU)

**Group 01-02 | DS261 Spring 2026**

This notebook implements a **PyTorch MLP** for binary classification of flight departure
delays (DEP_DEL15), running on the Databricks **GPU cluster**. It addresses every
limitation of the PySpark `MultilayerPerceptronClassifier`:

| Capability | PySpark MLP | This Notebook (PyTorch) |
|---|---|---|
| Activation | Sigmoid (fixed) | **ReLU** (avoids vanishing gradients) |
| Batch normalization | No | **Yes (BatchNorm1d)** |
| Dropout | No | **Yes (configurable)** |
| Optimizer | l-bfgs only | **Adam** with weight_decay (L2) |
| LR scheduling | No | **ReduceLROnPlateau** |
| Early stopping | `tol` only | **Validation-based** (patience=15) |
| Training curves | Not exposed | **Per-epoch train + val loss** |
| GPU | No | **Yes** |

**Preprocessing**: Same StandardScaler + PCA pipeline as the PySpark notebook (fit on
train only). Data flows: Spark parquets → VectorAssembler → StandardScaler → pandas →
PyTorch tensors.

**Architectures**: 3 configs matching the PySpark notebook for direct comparison, plus
dropout ablation and feature family ablation.

In [0]:
import json
import time
import os
import copy
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    confusion_matrix,
)

import pyspark.sql.functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import udf

# ── Configuration ──────────────────────────────────────────────────────────────
BASE_GROUP     = "dbfs:/student-groups/Group_01_02"
TARGET         = "DEP_DEL15"
BETA           = 2.0
SEED           = 42

TRAIN_FE_PATH  = f"{BASE_GROUP}/df_train_phase3.parquet"
VAL_FE_PATH    = f"{BASE_GROUP}/df_val_phase3.parquet"
TEST_FE_PATH   = f"{BASE_GROUP}/df_test_phase3.parquet"
NUM_COLS_PATH  = f"{BASE_GROUP}/phase3_final_num_cols.json"

RESULTS_CSV    = f"{BASE_GROUP}/phase3_results_summary_mlp_pytorch.csv"
CHART_DIR      = "/dbfs/FileStore/group_01_02/phase3_charts"

UNDERSAMPLE_RATIO = 2.0
PCA_VARIANCE_TARGET = 0.95

# Training hyperparameters
BATCH_SIZE     = 4096
MAX_EPOCHS     = 200
LEARNING_RATE  = 0.001
WEIGHT_DECAY   = 1e-4    # L2 regularization via Adam
DROPOUT_RATE   = 0.3
PATIENCE       = 15      # early stopping patience

# Auto-detect GPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"F-β metric: β = {BETA}  (recall weighted {int(BETA)}× over precision)")

PyTorch version: 2.7.0+cu126
Device: cuda
GPU: Tesla T4
GPU Memory: 15.7 GB
F-β metric: β = 2.0  (recall weighted 2× over precision)


## 1. Data Loading & Preprocessing

Load Phase 3.3 feature-engineered parquets, apply VectorAssembler + StandardScaler
(fit on train only), then convert to PyTorch tensors.

In [0]:
# Load feature manifest
num_cols_raw = dbutils.fs.head(NUM_COLS_PATH, 100000)
numerical_cols = json.loads(num_cols_raw)
N_FEATURES = len(numerical_cols)
print(f"Feature manifest: {N_FEATURES} features")

# Load parquets
df_train_fe = spark.read.parquet(TRAIN_FE_PATH).cache()
df_val_fe   = spark.read.parquet(VAL_FE_PATH).cache()
df_test_fe  = spark.read.parquet(TEST_FE_PATH).cache()

# Verify columns
missing = [c for c in numerical_cols if c not in df_train_fe.columns]
if missing:
    print(f"WARNING: {len(missing)} columns missing: {missing[:10]}")
    numerical_cols = [c for c in numerical_cols if c in df_train_fe.columns]
    N_FEATURES = len(numerical_cols)
else:
    print(f"All {N_FEATURES} columns present.")

n_train = df_train_fe.count()
print(f"Train rows: {n_train:,}")

Feature manifest: 104 features
All 104 columns present.
Train rows: 18,698,999


In [0]:
# ── VectorAssembler + StandardScaler (fit on train only) ─────────────────────
to_dense_udf = udf(lambda v: Vectors.dense(v.toArray()) if v is not None else None, VectorUDT())

# Null audit: any null in the feature columns will cause silent row drops.
# Phase 3.3 should have imputed all nulls; assert here so failures are visible.
_null_counts = {c: df_train_fe.filter(F.col(c).isNull()).count() for c in numerical_cols}
_cols_with_nulls = {c: n for c, n in _null_counts.items() if n > 0}
if _cols_with_nulls:
    print(f"WARNING: {len(_cols_with_nulls)} feature column(s) have nulls — re-run Phase 3.3 imputation:")
    for c, n in sorted(_cols_with_nulls.items(), key=lambda x: -x[1])[:10]:
        print(f"  {c}: {n:,} nulls")
else:
    print("Null audit passed: no nulls in feature columns.")

assembler = VectorAssembler(
    inputCols=numerical_cols,
    outputCol="features_sparse",
    handleInvalid="error",   # fail loudly if nulls reach assembly after imputation
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True,
)

def assemble_and_densify(df):
    return assembler.transform(df).withColumn("features_raw", to_dense_udf("features_sparse")).drop("features_sparse")

print("Fitting StandardScaler on train...")
t0 = time.time()
df_train_assembled = assemble_and_densify(df_train_fe).cache()
scaler_model = scaler.fit(df_train_assembled)
print(f"StandardScaler fit: {time.time()-t0:.1f}s")

def scale_split(df):
    assembled = assemble_and_densify(df)
    return scaler_model.transform(assembled)

df_train_scaled = scale_split(df_train_fe).cache()
df_val_scaled   = scale_split(df_val_fe).cache()
df_test_scaled  = scale_split(df_test_fe).cache()

df_train_assembled.unpersist()
_dim = df_train_scaled.first()["features"].size
print(f"Feature vector size: {_dim}")

Null audit passed: no nulls in feature columns.
Fitting StandardScaler on train...
StandardScaler fit: 1451.2s
Feature vector size: 104


## 2. PCA (for later experiment)

In [0]:
# Fit PCA on StandardScaled train
df_train_std_select = df_train_scaled.select("features", TARGET).cache()

pca_full = PCA(k=N_FEATURES, inputCol="features", outputCol="pca_features_full")
pca_full_model = pca_full.fit(df_train_std_select)
explained_variance = pca_full_model.explainedVariance.toArray()
cumulative_variance = np.cumsum(explained_variance)
k_target = int(np.searchsorted(cumulative_variance, PCA_VARIANCE_TARGET) + 1)
k_target = min(k_target, N_FEATURES)
print(f"PCA: {k_target} components for {PCA_VARIANCE_TARGET*100:.0f}% variance "
      f"(actual: {cumulative_variance[k_target-1]*100:.1f}%)")

pca_reduced = PCA(k=k_target, inputCol="features", outputCol="pca_features")
pca_model = pca_reduced.fit(df_train_std_select)

# Save PCA variance chart
os.makedirs(CHART_DIR, exist_ok=True)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, N_FEATURES+1), cumulative_variance*100, "b-", lw=2)
ax.axhline(y=PCA_VARIANCE_TARGET*100, color="r", ls="--", label=f"{PCA_VARIANCE_TARGET*100:.0f}% threshold")
ax.axvline(x=k_target, color="g", ls="--", label=f"k={k_target}")
ax.set_xlabel("Components"); ax.set_ylabel("Cumulative Variance (%)")
ax.set_title("PCA: Cumulative Explained Variance"); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(f"{CHART_DIR}/pca_variance_explained.png", dpi=150); plt.close(fig)
print(f"Saved: {CHART_DIR}/pca_variance_explained.png")

PCA: 55 components for 95% variance (actual: 95.2%)
Saved: /dbfs/FileStore/group_01_02/phase3_charts/pca_variance_explained.png


## 3. Undersampling & Spark → PyTorch Conversion

Convert Spark DataFrames to numpy arrays on the driver, then to PyTorch tensors.
The undersampled train is ~10M rows; we auto-detect available memory and subsample
if needed.

In [0]:
# Undersample train
n_delayed = df_train_scaled.filter(F.col(TARGET) == 1).count()
n_ontime  = df_train_scaled.filter(F.col(TARGET) == 0).count()
target_ontime = int(UNDERSAMPLE_RATIO * n_delayed)

print(f"Train before undersampling: on-time={n_ontime:,}  delayed={n_delayed:,}")

if target_ontime < n_ontime:
    fraction = target_ontime / n_ontime
    df_ontime_sample  = df_train_scaled.filter(F.col(TARGET) == 0).sample(fraction=fraction, seed=SEED)
    df_delayed_sample = df_train_scaled.filter(F.col(TARGET) == 1)
    df_train_us = df_ontime_sample.union(df_delayed_sample).cache()
else:
    df_train_us = df_train_scaled

n_us = df_train_us.count()
print(f"After undersampling: {n_us:,} rows")

# Auto-size: limit to ~2M rows for toPandas if dataset is huge
MAX_ROWS_FOR_TOPANDAS = 2_000_000
if n_us > MAX_ROWS_FOR_TOPANDAS:
    sample_frac = MAX_ROWS_FOR_TOPANDAS / n_us
    print(f"Subsampling to ~{MAX_ROWS_FOR_TOPANDAS:,} rows for driver memory (frac={sample_frac:.4f})")
    df_train_us = df_train_us.sample(fraction=sample_frac, seed=SEED).cache()
    n_us = df_train_us.count()
    print(f"Final train size: {n_us:,}")

Train before undersampling: on-time=15,069,828  delayed=3,629,171
After undersampling: 10,887,234 rows
Subsampling to ~2,000,000 rows for driver memory (frac=0.1837)
Final train size: 2,002,691


In [0]:
def spark_to_numpy(df, feature_col="features", label_col=TARGET, max_rows=None):
    """Convert Spark DataFrame with features vector to numpy arrays."""
    sel = df.withColumn("features_array", vector_to_array(feature_col)).select("features_array", label_col)
    if max_rows:
        sel = sel.limit(max_rows)
    pdf = sel.toPandas()
    X = np.array(pdf["features_array"].tolist(), dtype=np.float32)
    y = pdf[label_col].values.astype(np.int64)
    return X, y

print("Converting train to numpy...")
t0 = time.time()
X_train, y_train = spark_to_numpy(df_train_us)
print(f"  Train: {X_train.shape} ({time.time()-t0:.1f}s)")

print("Converting val to numpy...")
t0 = time.time()
X_val, y_val = spark_to_numpy(df_val_scaled, max_rows=500_000)
print(f"  Val:   {X_val.shape} ({time.time()-t0:.1f}s)")

print("Converting test to numpy...")
t0 = time.time()
X_test, y_test = spark_to_numpy(df_test_scaled, max_rows=500_000)
print(f"  Test:  {X_test.shape} ({time.time()-t0:.1f}s)")

# Also prepare PCA versions
print("Converting PCA train to numpy...")
df_train_pca = pca_model.transform(df_train_us.select("features", TARGET)).select("pca_features", TARGET)
df_train_pca = df_train_pca.withColumnRenamed("pca_features", "features")
X_train_pca, _ = spark_to_numpy(df_train_pca)

df_val_pca = pca_model.transform(df_val_scaled.select("features", TARGET)).select("pca_features", TARGET)
df_val_pca = df_val_pca.withColumnRenamed("pca_features", "features")
X_val_pca, _ = spark_to_numpy(df_val_pca, max_rows=500_000)

df_test_pca = pca_model.transform(df_test_scaled.select("features", TARGET)).select("pca_features", TARGET)
df_test_pca = df_test_pca.withColumnRenamed("pca_features", "features")
X_test_pca, _ = spark_to_numpy(df_test_pca, max_rows=500_000)

print(f"  PCA Train: {X_train_pca.shape}")

# Class balance
print(f"\nClass balance (train): {y_train.mean():.3f} delayed ({y_train.sum():,} / {len(y_train):,})")

Converting train to numpy...
  Train: (2002691, 104) (32.3s)
Converting val to numpy...
  Val:   (500000, 104) (549.0s)
Converting test to numpy...
  Test:  (500000, 104) (530.2s)
Converting PCA train to numpy...
  PCA Train: (2002691, 55)

Class balance (train): 0.333 delayed (667,770 / 2,002,691)


In [0]:
# ── Create DataLoaders ───────────────────────────────────────────────────────

def make_dataloader(X, y, batch_size=BATCH_SIZE, shuffle=True):
    dataset = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True)

train_loader = make_dataloader(X_train, y_train, shuffle=True)
val_loader   = make_dataloader(X_val, y_val, shuffle=False)
test_loader  = make_dataloader(X_test, y_test, shuffle=False)

print(f"DataLoaders created: {len(train_loader)} train batches, "
      f"{len(val_loader)} val batches, {len(test_loader)} test batches")

DataLoaders created: 489 train batches, 123 val batches, 123 test batches


## 4. Model Definition

PyTorch MLP with **BatchNorm**, **Dropout**, and **ReLU** activations.
- **ReLU** avoids vanishing gradients (PySpark MLP is stuck with sigmoid)
- **BatchNorm1d** normalizes activations per-batch, stabilizing training
- **Dropout** randomly zeroes neurons during training (PySpark and sklearn lack this)

In [0]:
class FlightDelayMLP(nn.Module):
    """MLP for binary classification with BatchNorm + Dropout."""

    def __init__(self, input_dim, hidden_dims, dropout_rate=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
            ])
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, 2))  # 2-class output
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

print("FlightDelayMLP defined.")

FlightDelayMLP defined.


## 5. Training & Evaluation Functions

Training loop with:
- **Adam optimizer** with weight_decay (L2 regularization)
- **ReduceLROnPlateau** learning rate scheduler
- **Early stopping** based on validation loss (patience=15)
- **Per-epoch logging** of train loss, val loss, val F2, val AUROC

In [0]:
def fbeta(precision, recall, beta=BETA):
    denom = beta**2 * precision + recall
    if denom < 1e-12:
        return 0.0
    return (1 + beta**2) * precision * recall / denom


def evaluate_pytorch(model, dataloader, device, threshold=0.5):
    """Evaluate model on a DataLoader, return metrics dict."""
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            logits = model(X_batch.to(device))
            probs = torch.softmax(logits, dim=1)[:, 1]
            all_probs.append(probs.cpu())
            all_labels.append(y_batch)

    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    preds = (probs >= threshold).astype(int)

    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    fb = fbeta(prec, rec)
    auroc = roc_auc_score(labels, probs)
    acc = (preds == labels).mean()

    tp = int(((preds == 1) & (labels == 1)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())
    tn = int(((preds == 0) & (labels == 0)).sum())

    return {
        f"F{BETA:.0f} (delayed)": round(fb, 4),
        "F1 (delayed)": round(f1, 4),
        "Precision (delayed)": round(prec, 4),
        "Recall (delayed)": round(rec, 4),
        "AUROC": round(auroc, 4),
        "Accuracy": round(acc, 4),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    }


def print_metrics(metrics, title=""):
    if title:
        print(f"\n{'='*60}")
        print(f"  {title}")
        print(f"{'='*60}")
    for k, v in metrics.items():
        if k in ("TP", "FP", "FN", "TN"):
            print(f"  {k:25s}: {v:>12,}")
        else:
            marker = "  <-- PRIMARY" if f"F{BETA:.0f}" in k and "delayed" in k else ""
            print(f"  {k:25s}: {v:>12.4f}{marker}")


def train_model(model, train_loader, val_loader, device,
                max_epochs=MAX_EPOCHS, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
                patience=PATIENCE, verbose=True):
    """
    Train a PyTorch model with Adam, LR scheduling, and early stopping.
    Returns: trained model, training history dict.
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=5, factor=0.5
    )

    history = {
        "train_loss": [], "val_loss": [],
        "val_f2": [], "val_auroc": [],
        "lr": [],
    }

    best_val_loss = float("inf")
    best_model_state = None
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        # ── Train ────────────────────────────────────────────────────
        model.train()
        train_loss_sum = 0.0
        n_batches = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item()
            n_batches += 1

        avg_train_loss = train_loss_sum / n_batches

        # ── Validate ─────────────────────────────────────────────────
        model.eval()
        val_loss_sum = 0.0
        val_batches = 0
        all_probs, all_labels = [], []

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                val_loss_sum += loss.item()
                val_batches += 1

                probs = torch.softmax(logits, dim=1)[:, 1]
                all_probs.append(probs.cpu())
                all_labels.append(y_batch.cpu())

        avg_val_loss = val_loss_sum / val_batches
        probs_np = torch.cat(all_probs).numpy()
        labels_np = torch.cat(all_labels).numpy()
        preds_np = (probs_np >= 0.5).astype(int)

        prec = precision_score(labels_np, preds_np, zero_division=0)
        rec = recall_score(labels_np, preds_np, zero_division=0)
        val_f2 = fbeta(prec, rec)
        val_auroc = roc_auc_score(labels_np, probs_np)

        current_lr = optimizer.param_groups[0]["lr"]
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["val_f2"].append(val_f2)
        history["val_auroc"].append(val_auroc)
        history["lr"].append(current_lr)

        # LR scheduling
        scheduler.step(avg_val_loss)

        # Early stopping
        if avg_val_loss < best_val_loss - 1e-6:
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if verbose and (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1:>3d}/{max_epochs}: "
                  f"train_loss={avg_train_loss:.4f}  val_loss={avg_val_loss:.4f}  "
                  f"F2={val_f2:.4f}  AUROC={val_auroc:.4f}  lr={current_lr:.6f}"
                  f"{'  *best*' if epochs_no_improve == 0 else ''}")

        if epochs_no_improve >= patience:
            if verbose:
                print(f"  Early stopping at epoch {epoch+1} (patience={patience})")
            break

    # Restore best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    n_epochs = len(history["train_loss"])
    if verbose:
        print(f"  Training complete: {n_epochs} epochs, best val_loss={best_val_loss:.4f}")

    return model, history


print("Training and evaluation functions defined.")

Training and evaluation functions defined.


## 6. Architecture A: `(64,)` — Shallow Baseline

In [0]:
LAYERS_A = (64,)
model_a = FlightDelayMLP(N_FEATURES, LAYERS_A, dropout_rate=DROPOUT_RATE)
print(f"Architecture A: hidden={LAYERS_A}, params={model_a.count_parameters():,}")

print("Training PyTorch MLP-A...")
t0 = time.time()
model_a, history_a = train_model(model_a, train_loader, val_loader, DEVICE)
time_a = time.time() - t0
print(f"Done in {time_a:.1f}s")

metrics_a = evaluate_pytorch(model_a, val_loader, DEVICE)
print_metrics(metrics_a, f"PyTorch-A {LAYERS_A} — VALIDATION")

Architecture A: hidden=(64,), params=6,978
Training PyTorch MLP-A...
  Epoch   5/200: train_loss=0.5783  val_loss=0.5509  F2=0.4118  AUROC=0.6822  lr=0.001000
  Epoch  10/200: train_loss=0.5767  val_loss=0.5594  F2=0.4238  AUROC=0.6826  lr=0.000500
  Epoch  15/200: train_loss=0.5760  val_loss=0.5619  F2=0.4252  AUROC=0.6823  lr=0.000250
  Early stopping at epoch 16 (patience=15)
  Training complete: 16 epochs, best val_loss=0.5334
Done in 505.1s

  PyTorch-A (64,) — VALIDATION
  F2 (delayed)             :       0.3918  <-- PRIMARY
  F1 (delayed)             :       0.3481
  Precision (delayed)      :       0.2936
  Recall (delayed)         :       0.4275
  AUROC                    :       0.6819
  Accuracy                 :       0.7426
  TP                       :       34,369
  FP                       :       82,685
  FN                       :       46,017
  TN                       :      336,929


## 7. Architecture B: `(128, 64)` — Two Hidden Layers

In [0]:
LAYERS_B = (128, 64)
model_b = FlightDelayMLP(N_FEATURES, LAYERS_B, dropout_rate=DROPOUT_RATE)
print(f"Architecture B: hidden={LAYERS_B}, params={model_b.count_parameters():,}")

print("Training PyTorch MLP-B...")
t0 = time.time()
model_b, history_b = train_model(model_b, train_loader, val_loader, DEVICE)
time_b = time.time() - t0
print(f"Done in {time_b:.1f}s")

metrics_b = evaluate_pytorch(model_b, val_loader, DEVICE)
print_metrics(metrics_b, f"PyTorch-B {LAYERS_B} — VALIDATION")

Architecture B: hidden=(128, 64), params=22,210
Training PyTorch MLP-B...
  Epoch   5/200: train_loss=0.5769  val_loss=0.5503  F2=0.4242  AUROC=0.6835  lr=0.001000
  Epoch  10/200: train_loss=0.5735  val_loss=0.5524  F2=0.4261  AUROC=0.6857  lr=0.000500
  Epoch  15/200: train_loss=0.5719  val_loss=0.5489  F2=0.4229  AUROC=0.6869  lr=0.000250
  Early stopping at epoch 16 (patience=15)
  Training complete: 16 epochs, best val_loss=0.5336
Done in 527.5s

  PyTorch-B (128, 64) — VALIDATION
  F2 (delayed)             :       0.4057  <-- PRIMARY
  F1 (delayed)             :       0.3512
  Precision (delayed)      :       0.2870
  Recall (delayed)         :       0.4524
  AUROC                    :       0.6815
  Accuracy                 :       0.7313
  TP                       :       36,366
  FP                       :       90,332
  FN                       :       44,020
  TN                       :      329,282


## 8. Architecture C: `(256, 128)` — Wider Network

In [0]:
LAYERS_C = (256, 128)
model_c = FlightDelayMLP(N_FEATURES, LAYERS_C, dropout_rate=DROPOUT_RATE)
print(f"Architecture C: hidden={LAYERS_C}, params={model_c.count_parameters():,}")

print("Training PyTorch MLP-C...")
t0 = time.time()
model_c, history_c = train_model(model_c, train_loader, val_loader, DEVICE)
time_c = time.time() - t0
print(f"Done in {time_c:.1f}s")

metrics_c = evaluate_pytorch(model_c, val_loader, DEVICE)
print_metrics(metrics_c, f"PyTorch-C {LAYERS_C} — VALIDATION")

Architecture C: hidden=(256, 128), params=60,802
Training PyTorch MLP-C...
  Epoch   5/200: train_loss=0.5748  val_loss=0.5509  F2=0.4262  AUROC=0.6850  lr=0.001000
  Epoch  10/200: train_loss=0.5723  val_loss=0.5410  F2=0.4177  AUROC=0.6876  lr=0.001000
  Epoch  15/200: train_loss=0.5710  val_loss=0.5438  F2=0.4163  AUROC=0.6863  lr=0.001000
  Epoch  20/200: train_loss=0.5683  val_loss=0.5469  F2=0.4247  AUROC=0.6887  lr=0.000500
  Epoch  25/200: train_loss=0.5659  val_loss=0.5526  F2=0.4255  AUROC=0.6883  lr=0.000250
  Early stopping at epoch 26 (patience=15)
  Training complete: 26 epochs, best val_loss=0.5245
Done in 832.7s

  PyTorch-C (256, 128) — VALIDATION
  F2 (delayed)             :       0.3902  <-- PRIMARY
  F1 (delayed)             :       0.3527
  Precision (delayed)      :       0.3040
  Recall (delayed)         :       0.4200
  AUROC                    :       0.6882
  Accuracy                 :       0.7522
  TP                       :       33,763
  FP                

## 9. Architecture Comparison

In [0]:
f2_key = f"F{BETA:.0f} (delayed)"
arch_results = {
    f"PyTorch-A {LAYERS_A}": {"metrics": metrics_a, "time": time_a, "model": model_a, "history": history_a, "layers": LAYERS_A},
    f"PyTorch-B {LAYERS_B}": {"metrics": metrics_b, "time": time_b, "model": model_b, "history": history_b, "layers": LAYERS_B},
    f"PyTorch-C {LAYERS_C}": {"metrics": metrics_c, "time": time_c, "model": model_c, "history": history_c, "layers": LAYERS_C},
}

print(f"\n{'='*80}")
print(f"  ARCHITECTURE COMPARISON — VALIDATION (PyTorch, StandardScaler, GPU)")
print(f"{'='*80}")
print(f"  {'Architecture':<25s} {'F2':>8s} {'F1':>8s} {'Prec':>8s} {'Rec':>8s} {'AUROC':>8s} {'Epochs':>8s} {'Time':>8s}")
print(f"  {'-'*85}")

best_arch_name = None
best_f2 = -1.0

for name, data in arch_results.items():
    m = data["metrics"]
    n_ep = len(data["history"]["train_loss"])
    print(f"  {name:<25s} {m[f2_key]:>8.4f} {m['F1 (delayed)']:>8.4f} "
          f"{m['Precision (delayed)']:>8.4f} {m['Recall (delayed)']:>8.4f} "
          f"{m['AUROC']:>8.4f} {n_ep:>8d} {data['time']:>7.1f}s")
    if m[f2_key] > best_f2:
        best_f2 = m[f2_key]
        best_arch_name = name

print(f"\n  Best: {best_arch_name} (F2={best_f2:.4f})")
best_model = arch_results[best_arch_name]["model"]
best_history = arch_results[best_arch_name]["history"]
best_layers = arch_results[best_arch_name]["layers"]
best_time = arch_results[best_arch_name]["time"]


  ARCHITECTURE COMPARISON — VALIDATION (PyTorch, StandardScaler, GPU)
  Architecture                    F2       F1     Prec      Rec    AUROC   Epochs     Time
  -------------------------------------------------------------------------------------
  PyTorch-A (64,)             0.3918   0.3481   0.2936   0.4275   0.6819       16   505.1s
  PyTorch-B (128, 64)         0.4057   0.3512   0.2870   0.4524   0.6815       16   527.5s
  PyTorch-C (256, 128)        0.3902   0.3527   0.3040   0.4200   0.6882       26   832.7s

  Best: PyTorch-B (128, 64) (F2=0.4057)


In [0]:
# Architecture comparison chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
short_names = [f"A: {LAYERS_A}", f"B: {LAYERS_B}", f"C: {LAYERS_C}"]
f2_vals = [arch_results[n]["metrics"][f2_key] for n in arch_results]
auroc_vals = [arch_results[n]["metrics"]["AUROC"] for n in arch_results]
colors = ["#2196F3", "#FF9800", "#4CAF50"]

for ax, vals, ylabel, title in [
    (axes[0], f2_vals, "F2 Score (β=2)", "F2"),
    (axes[1], auroc_vals, "AUROC", "AUROC"),
]:
    bars = ax.bar(short_names, vals, color=colors, edgecolor="black", lw=0.5)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(f"PyTorch MLP — {title}", fontsize=13)
    ax.set_ylim(min(vals)*0.9, max(vals)*1.05)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                f"{val:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

fig.suptitle("PyTorch MLP Architecture Comparison (Validation)", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(f"{CHART_DIR}/pytorch_architecture_comparison.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {CHART_DIR}/pytorch_architecture_comparison.png")

Saved: /dbfs/FileStore/group_01_02/phase3_charts/pytorch_architecture_comparison.png


## 10. Training Curves

Per-epoch train and validation loss for the best architecture. This is the **real
convergence proof** — PySpark MLP cannot provide this.

In [0]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(best_history["train_loss"]) + 1)

# Loss curves
ax1.plot(epochs, best_history["train_loss"], "b-", lw=2, label="Train Loss")
ax1.plot(epochs, best_history["val_loss"], "r-", lw=2, label="Val Loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Cross-Entropy Loss")
ax1.set_title(f"Training Curves — PyTorch {best_layers}"); ax1.legend(); ax1.grid(True, alpha=0.3)

# F2 + AUROC over epochs
ax2.plot(epochs, best_history["val_f2"], "g-", lw=2, label="Val F2")
ax2.plot(epochs, best_history["val_auroc"], "m--", lw=2, label="Val AUROC")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Score")
ax2.set_title(f"Validation Metrics — PyTorch {best_layers}"); ax2.legend(); ax2.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(f"{CHART_DIR}/pytorch_training_curves.png", dpi=150)
plt.close(fig)
print(f"Saved: {CHART_DIR}/pytorch_training_curves.png")

Saved: /dbfs/FileStore/group_01_02/phase3_charts/pytorch_training_curves.png


## 11. Dropout Ablation

Test the impact of dropout rate on model performance. Dropout is a regularization
technique that PySpark and sklearn MLP both lack — this experiment shows its value.

In [0]:
dropout_rates = [0.0, 0.1, 0.2, 0.3, 0.5]
dropout_results = []

print(f"\n{'='*60}")
print(f"  DROPOUT ABLATION — PyTorch {best_layers}")
print(f"{'='*60}")
print(f"  {'Dropout':>8s} {'F2':>8s} {'AUROC':>8s} {'Epochs':>8s} {'Time':>8s}")
print(f"  {'-'*40}")

for dr in dropout_rates:
    m = FlightDelayMLP(N_FEATURES, best_layers, dropout_rate=dr).to(DEVICE)
    t0 = time.time()
    m, h = train_model(m, train_loader, val_loader, DEVICE, verbose=False)
    t_elapsed = time.time() - t0
    metrics = evaluate_pytorch(m, val_loader, DEVICE)
    n_ep = len(h["train_loss"])
    dropout_results.append({
        "dropout": dr, **metrics, "epochs": n_ep, "time": t_elapsed
    })
    print(f"  {dr:>8.1f} {metrics[f2_key]:>8.4f} {metrics['AUROC']:>8.4f} "
          f"{n_ep:>8d} {t_elapsed:>7.1f}s")

# Chart
fig, ax = plt.subplots(figsize=(10, 6))
drs = [r["dropout"] for r in dropout_results]
f2s = [r[f2_key] for r in dropout_results]
bars = ax.bar([str(d) for d in drs], f2s, color="#9C27B0", edgecolor="black", lw=0.5)
ax.set_xlabel("Dropout Rate"); ax.set_ylabel("F2 Score (validation)")
ax.set_title(f"Dropout Ablation — PyTorch {best_layers}"); ax.grid(True, axis="y", alpha=0.3)
for bar, val in zip(bars, f2s):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f"{val:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{CHART_DIR}/pytorch_dropout_ablation.png", dpi=150)
plt.close(fig)
print(f"Saved: {CHART_DIR}/pytorch_dropout_ablation.png")


  DROPOUT ABLATION — PyTorch (128, 64)
   Dropout       F2    AUROC   Epochs     Time
  ----------------------------------------
       0.0   0.3962   0.6824       16   516.5s
       0.1   0.4058   0.6820       16   511.4s
       0.2   0.3970   0.6875       29   945.8s
       0.3   0.4057   0.6862       36  1155.3s
       0.5   0.3957   0.6823       19   618.4s
Saved: /dbfs/FileStore/group_01_02/phase3_charts/pytorch_dropout_ablation.png


## 12. PCA Experiment

In [0]:
# Create PCA DataLoaders
train_pca_loader = make_dataloader(X_train_pca, y_train, shuffle=True)
val_pca_loader = make_dataloader(X_val_pca, y_val, shuffle=False)

pca_model_pt = FlightDelayMLP(k_target, best_layers, dropout_rate=DROPOUT_RATE)
print(f"PCA model: input={k_target}, hidden={best_layers}, params={pca_model_pt.count_parameters():,}")

t0 = time.time()
pca_model_pt, pca_history = train_model(pca_model_pt, train_pca_loader, val_pca_loader, DEVICE)
pca_time = time.time() - t0

pca_metrics = evaluate_pytorch(pca_model_pt, val_pca_loader, DEVICE)
print_metrics(pca_metrics, f"PyTorch-PCA {best_layers} (k={k_target}) — VALIDATION")
print(f"  StandardScaler best: F2={best_f2:.4f}")
print(f"  PCA ({k_target} comps):    F2={pca_metrics[f2_key]:.4f}")
print(f"  Delta:               {pca_metrics[f2_key] - best_f2:+.4f}")

# Decide final model
if pca_metrics[f2_key] > best_f2:
    final_model = pca_model_pt
    final_val_loader = val_pca_loader
    final_test_loader = make_dataloader(X_test_pca, y_test, shuffle=False)
    final_label = f"PyTorch-PCA {best_layers} (k={k_target})"
    best_time = pca_time
else:
    final_model = best_model
    final_val_loader = val_loader
    final_test_loader = test_loader
    final_label = best_arch_name

print(f"\nFinal model: {final_label}")

PCA model: input=55, hidden=(128, 64), params=15,938
  Epoch   5/200: train_loss=0.5789  val_loss=0.5764  F2=0.1800  AUROC=0.4992  lr=0.001000  *best*
  Epoch  10/200: train_loss=0.5765  val_loss=0.5754  F2=0.1743  AUROC=0.4992  lr=0.001000  *best*
  Epoch  15/200: train_loss=0.5757  val_loss=0.5771  F2=0.1813  AUROC=0.4993  lr=0.001000
  Epoch  20/200: train_loss=0.5752  val_loss=0.5759  F2=0.1772  AUROC=0.4995  lr=0.001000
  Epoch  25/200: train_loss=0.5750  val_loss=0.5715  F2=0.1685  AUROC=0.4996  lr=0.001000
  Epoch  30/200: train_loss=0.5736  val_loss=0.5698  F2=0.1698  AUROC=0.4996  lr=0.000500
  Early stopping at epoch 34 (patience=15)
  Training complete: 34 epochs, best val_loss=0.5650

  PyTorch-PCA (128, 64) (k=55) — VALIDATION
  F2 (delayed)             :       0.1621  <-- PRIMARY
  F1 (delayed)             :       0.1617
  Precision (delayed)      :       0.1612
  Recall (delayed)         :       0.1623
  AUROC                    :       0.4996
  Accuracy                 

## 13. Threshold Tuning

In [0]:
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
threshold_results = []
best_thresh = 0.5
best_thresh_f2 = -1.0

print(f"\n{'='*60}")
print(f"  THRESHOLD TUNING — {final_label}")
print(f"{'='*60}")
print(f"  {'Thresh':>7s} {'Prec':>8s} {'Rec':>8s} {'F1':>8s} {'F2':>8s} {'AUROC':>8s}")
print(f"  {'-'*50}")

for thresh in thresholds:
    m = evaluate_pytorch(final_model, final_val_loader, DEVICE, threshold=thresh)
    m["threshold"] = thresh
    threshold_results.append(m)
    f2_v = m[f2_key]
    print(f"  {thresh:>7.2f} {m['Precision (delayed)']:>8.4f} {m['Recall (delayed)']:>8.4f} "
          f"{m['F1 (delayed)']:>8.4f} {f2_v:>8.4f} {m['AUROC']:>8.4f}"
          f"{'  <-- best' if f2_v > best_thresh_f2 else ''}")
    if f2_v > best_thresh_f2:
        best_thresh_f2 = f2_v
        best_thresh = thresh

print(f"\n  Best threshold: {best_thresh} (F2={best_thresh_f2:.4f})")

# Chart
fig, ax = plt.subplots(figsize=(10, 6))
t_vals = [r["threshold"] for r in threshold_results]
ax.plot(t_vals, [r["Precision (delayed)"] for r in threshold_results], "s-", color="#E91E63", lw=2, ms=6, label="Precision")
ax.plot(t_vals, [r["Recall (delayed)"] for r in threshold_results], "o-", color="#2196F3", lw=2, ms=6, label="Recall")
ax.plot(t_vals, [r["F1 (delayed)"] for r in threshold_results], "^-", color="#FF9800", lw=2, ms=6, label="F1")
ax.plot(t_vals, [r[f2_key] for r in threshold_results], "D-", color="#4CAF50", lw=2.5, ms=7, label="F2 (primary)")
ax.axvline(x=best_thresh, color="gray", ls="--", alpha=0.7, label=f"Best={best_thresh}")
ax.set_xlabel("Threshold"); ax.set_ylabel("Score")
ax.set_title(f"Threshold Tuning — {final_label}"); ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f"{CHART_DIR}/pytorch_threshold_tuning.png", dpi=150)
plt.close(fig)
print(f"Saved: {CHART_DIR}/pytorch_threshold_tuning.png")


  THRESHOLD TUNING — PyTorch-B (128, 64)
   Thresh     Prec      Rec       F1       F2    AUROC
  --------------------------------------------------
     0.10   0.1624   0.9969   0.2793   0.4917   0.6815  <-- best
     0.15   0.1704   0.9734   0.2900   0.5011   0.6815  <-- best
     0.20   0.1828   0.9312   0.3057   0.5120   0.6815  <-- best
     0.25   0.1960   0.8776   0.3204   0.5176   0.6815  <-- best
     0.30   0.2092   0.8176   0.3332   0.5170   0.6815
     0.35   0.2237   0.7495   0.3446   0.5099   0.6815
     0.40   0.2396   0.6679   0.3527   0.4920   0.6815
     0.45   0.2597   0.5715   0.3571   0.4608   0.6815
     0.50   0.2870   0.4524   0.3512   0.4057   0.6815

  Best threshold: 0.25 (F2=0.5176)
Saved: /dbfs/FileStore/group_01_02/phase3_charts/pytorch_threshold_tuning.png


## 14. Feature Family Ablation

Remove each feature family and measure F2 impact. Unlike the sklearn ablation in the
PySpark notebook (300K subsample), this runs on the full PyTorch training set with
dropout and batch norm.

In [0]:
FEATURE_FAMILIES = {
    "Temporal": ["QUARTER", "MONTH", "DAY_OF_MONTH", "YEAR", "CRS_DEP_TIME", "dep_hour",
                 "is_morning_peak", "is_evening_peak", "is_weekend", "is_late_night",
                 "dep_hour_sin", "dep_hour_cos", "day_sin", "day_cos",
                 "month_sin", "month_cos", "quarter_sin", "quarter_cos"],
    "Holiday": ["is_holiday", "is_pre_holiday", "is_post_holiday", "is_holiday_weekend_adjacent"],
    "Congestion": ["origin_2h_sched_departures", "origin_sched_count_prev2h", "is_origin_busy_2h_before"],
    "Target Encoding": ["carrier_delay_rate", "origin_delay_rate", "dest_delay_rate",
                        "route_delay_rate", "hour_delay_rate"],
    "Weather": [c for c in numerical_cols if c.startswith("avg_Hourly") or c in
                ["precip_bin", "wind_bin", "vis_bin", "wx_evening_x_bad_vis", "wx_peak_x_strong_wind"]],
    "Airport Static": ["origin_train_departures", "dest_train_departures", "is_hub_origin",
                       "is_hub_dest", "origin_dest_distinct_count"],
    "Interactions": [c for c in numerical_cols if "_x_" in c and c not in
                     ["wx_evening_x_bad_vis", "wx_peak_x_strong_wind"]],
    "Indices": ["route_carrier_residual", "origin_pressure_index",
                "temp_anomaly_station_month", "severe_wx_index",
                "hour_month_cross_1", "hour_month_cross_2"],
}

family_indices = {}
for family, cols in FEATURE_FAMILIES.items():
    indices = [numerical_cols.index(c) for c in cols if c in numerical_cols]
    family_indices[family] = indices
    print(f"  {family:<20s}: {len(indices)} features")

  Temporal            : 18 features
  Holiday             : 4 features
  Congestion          : 3 features
  Target Encoding     : 5 features
  Weather             : 20 features
  Airport Static      : 5 features
  Interactions        : 38 features
  Indices             : 6 features


In [0]:
# Baseline
model_base = FlightDelayMLP(N_FEATURES, best_layers, dropout_rate=DROPOUT_RATE)
model_base, _ = train_model(model_base, train_loader, val_loader, DEVICE, verbose=False)
baseline_metrics = evaluate_pytorch(model_base, val_loader, DEVICE)
baseline_f2 = baseline_metrics[f2_key]
print(f"Baseline (all features): F2={baseline_f2:.4f}")

ablation_results = []
print(f"\n{'='*60}")
print(f"  FEATURE FAMILY ABLATION — PyTorch {best_layers}")
print(f"{'='*60}")
print(f"  {'Family Removed':<20s} {'F2':>8s} {'ΔF2':>8s} {'N':>5s}")
print(f"  {'-'*45}")

for family, indices in family_indices.items():
    if not indices:
        continue

    keep_mask = np.ones(N_FEATURES, dtype=bool)
    keep_mask[indices] = False
    n_keep = keep_mask.sum()

    X_train_abl = X_train[:, keep_mask]
    X_val_abl = X_val[:, keep_mask]
    abl_train_loader = make_dataloader(X_train_abl, y_train, shuffle=True)
    abl_val_loader = make_dataloader(X_val_abl, y_val, shuffle=False)

    m = FlightDelayMLP(n_keep, best_layers, dropout_rate=DROPOUT_RATE)
    m, _ = train_model(m, abl_train_loader, abl_val_loader, DEVICE, verbose=False)
    abl_metrics = evaluate_pytorch(m, abl_val_loader, DEVICE)
    abl_f2 = abl_metrics[f2_key]
    delta = abl_f2 - baseline_f2

    ablation_results.append({"family": family, "f2": abl_f2, "delta_f2": delta, "n_removed": len(indices)})
    print(f"  {family:<20s} {abl_f2:>8.4f} {delta:>+8.4f} {len(indices):>5d}")

# Chart
ablation_results.sort(key=lambda x: x["delta_f2"])
fig, ax = plt.subplots(figsize=(10, 6))
fams = [r["family"] for r in ablation_results]
deltas = [r["delta_f2"] for r in ablation_results]
bar_colors = ["#F44336" if d < 0 else "#4CAF50" for d in deltas]
ax.barh(fams, deltas, color=bar_colors, edgecolor="black", lw=0.5)
ax.axvline(x=0, color="black", lw=0.8)
ax.set_xlabel("ΔF2 (removing family)"); ax.set_title("Feature Ablation — PyTorch MLP")
for i, (f, d) in enumerate(zip(fams, deltas)):
    ax.text(d + (0.001 if d >= 0 else -0.001), i, f"{d:+.4f}",
            va="center", ha="left" if d >= 0 else "right", fontsize=9, fontweight="bold")
ax.grid(True, axis="x", alpha=0.3); fig.tight_layout()
fig.savefig(f"{CHART_DIR}/pytorch_feature_ablation.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {CHART_DIR}/pytorch_feature_ablation.png")

Baseline (all features): F2=0.4101

  FEATURE FAMILY ABLATION — PyTorch (128, 64)
  Family Removed             F2      ΔF2     N
  ---------------------------------------------
  Temporal               0.3575  -0.0526    18
  Holiday                0.4144  +0.0043     4
  Congestion             0.4013  -0.0088     3
  Target Encoding        0.3941  -0.0160     5
  Weather                0.4335  +0.0234    20
  Airport Static         0.4151  +0.0050     5
  Interactions           0.4184  +0.0083    38
  Indices                0.3951  -0.0150     6
Saved: /dbfs/FileStore/group_01_02/phase3_charts/pytorch_feature_ablation.png


## 15. Blind Test Evaluation

Final evaluation on the held-out 2019 test set with **frozen threshold**.

In [0]:
print(f"\n{'='*60}")
print(f"  BLIND TEST — {final_label}")
print(f"  Threshold: {best_thresh} (frozen from validation)")
print(f"{'='*60}")

test_metrics = evaluate_pytorch(final_model, final_test_loader, DEVICE, threshold=best_thresh)
print_metrics(test_metrics, f"{final_label} — TEST (BLIND)")

test_default = evaluate_pytorch(final_model, final_test_loader, DEVICE)
print_metrics(test_default, f"{final_label} — TEST (threshold=0.5)")


  BLIND TEST — PyTorch-B (128, 64)
  Threshold: 0.25 (frozen from validation)

  PyTorch-B (128, 64) — TEST (BLIND)
  F2 (delayed)             :       0.5244  <-- PRIMARY
  F1 (delayed)             :       0.3262
  Precision (delayed)      :       0.2001
  Recall (delayed)         :       0.8818
  AUROC                    :       0.6662
  Accuracy                 :       0.3855
  TP                       :       74,362
  FP                       :      297,296
  FN                       :        9,970
  TN                       :      118,372

  PyTorch-B (128, 64) — TEST (threshold=0.5)
  F2 (delayed)             :       0.3597  <-- PRIMARY
  F1 (delayed)             :       0.3337
  Precision (delayed)      :       0.2978
  Recall (delayed)         :       0.3794
  AUROC                    :       0.6662
  Accuracy                 :       0.7444
  TP                       :       31,993
  FP                       :       75,437
  FN                       :       52,339
  TN         

## 16. Results Summary & Save

In [0]:
results_rows = []

for name, data in arch_results.items():
    m = data["metrics"]
    results_rows.append({
        "Model": f"{name} (PyTorch, GPU)",
        "Split": "Validation",
        **{k: v for k, v in m.items()},
        "Train Time (s)": round(data["time"], 1),
    })

# PCA
results_rows.append({
    "Model": f"PyTorch-PCA {best_layers} (k={k_target})",
    "Split": "Validation",
    **{k: v for k, v in pca_metrics.items()},
    "Train Time (s)": round(pca_time, 1),
})

# Best with tuned threshold (val)
best_thresh_val = evaluate_pytorch(final_model, final_val_loader, DEVICE, threshold=best_thresh)
results_rows.append({
    "Model": f"{final_label} (thresh={best_thresh})",
    "Split": "Validation",
    **{k: v for k, v in best_thresh_val.items()},
    "Train Time (s)": round(best_time, 1),
})

# Blind test
results_rows.append({
    "Model": f"{final_label} (thresh={best_thresh})",
    "Split": "Test (Blind)",
    **{k: v for k, v in test_metrics.items()},
    "Train Time (s)": "",
})

results_df = pd.DataFrame(results_rows)
print("\nPyTorch MLP Results Summary:")
print(results_df.to_string(index=False))

# Save CSV
csv_cols = ["Model", "Split", f"F{BETA:.0f} (delayed)", "F1 (delayed)",
            "Precision (delayed)", "Recall (delayed)", "AUROC", "Accuracy",
            "TP", "FP", "FN", "TN", "Train Time (s)"]
results_csv = results_df[[c for c in csv_cols if c in results_df.columns]]
csv_local = "/tmp/phase3_results_summary_mlp_pytorch.csv"
results_csv.to_csv(csv_local, index=False)
dbutils.fs.cp(f"file:{csv_local}", RESULTS_CSV)
print(f"Saved: {RESULTS_CSV}")


PyTorch MLP Results Summary:
                              Model        Split  F2 (delayed)  F1 (delayed)  Precision (delayed)  Recall (delayed)  AUROC  Accuracy    TP     FP    FN     TN Train Time (s)
     PyTorch-A (64,) (PyTorch, GPU)   Validation        0.3918        0.3481               0.2936            0.4275 0.6819    0.7426 34369  82685 46017 336929          505.1
 PyTorch-B (128, 64) (PyTorch, GPU)   Validation        0.4057        0.3512               0.2870            0.4524 0.6815    0.7313 36366  90332 44020 329282          527.5
PyTorch-C (256, 128) (PyTorch, GPU)   Validation        0.3902        0.3527               0.3040            0.4200 0.6882    0.7522 33763  77294 46623 342320          832.7
       PyTorch-PCA (128, 64) (k=55)   Validation        0.1621        0.1617               0.1612            0.1623 0.4996    0.7295 13047  67897 67339 351717         1109.8
  PyTorch-B (128, 64) (thresh=0.25)   Validation        0.5176        0.3204               0.1960   

In [0]:
# ── Load existing results for comparison ─────────────────────────────────────

print("\n" + "="*80)
print("  ALL MODELS COMPARISON (including PyTorch)")
print("="*80)

existing = {
    "LR/RF": f"{BASE_GROUP}/phase3_results_summary_lr_rf.csv",
    "GBT": f"{BASE_GROUP}/phase3_results_summary_gbt.csv",
    "PySpark MLP": f"{BASE_GROUP}/phase3_results_summary_mlp.csv",
}

all_dfs = []
for label, path in existing.items():
    try:
        csv_text = dbutils.fs.head(path, 100000)
        all_dfs.append(pd.read_csv(StringIO(csv_text)))
        print(f"  Loaded {label}")
    except Exception as e:
        print(f"  {label}: {e}")

all_dfs.append(results_csv)
combined = pd.concat(all_dfs, ignore_index=True)

for split in ["Validation", "Test (Blind)"]:
    sub = combined[combined["Split"] == split]
    if sub.empty:
        continue
    print(f"\n  {split}:")
    print(f"  {'Model':<50s} {'F2':>7s} {'Rec':>7s} {'AUROC':>7s}")
    print(f"  {'-'*75}")
    for _, row in sub.iterrows():
        print(f"  {str(row['Model']):<50s} "
              f"{row.get(f'F{BETA:.0f} (delayed)', ''):>7} "
              f"{row.get('Recall (delayed)', ''):>7} "
              f"{row.get('AUROC', ''):>7}")


  ALL MODELS COMPARISON (including PyTorch)
  Loaded LR/RF
  Loaded GBT
  Loaded PySpark MLP

  Validation:
  Model                                                   F2     Rec   AUROC
  ---------------------------------------------------------------------------
  LR (grid search best)                               0.5077  0.8098  0.6624
  RF (undersampled, thresh=0.4)                       0.5147  0.8544  0.6741
  GBT (grid search, thresh=0.25)                      0.5182  0.8582  0.6828
  MLP-A [103, 64, 2] (StandardScaler)                 0.4699   0.642  0.6393
  MLP-B [103, 128, 64, 2] (StandardScaler)            0.4324  0.5198  0.6702
  MLP-C [103, 256, 128, 2] (StandardScaler)           0.4512  0.5711  0.6647
  MLP-PCA [54, 64, 2] (StdScaler+PCA)                  0.362  0.3874  0.6202
  MLP-A [103, 64, 2] (thresh=0.35)                    0.5068   0.846  0.6393
  PyTorch-A (64,) (PyTorch, GPU)                      0.3918  0.4275  0.6819
  PyTorch-B (128, 64) (PyTorch, GPU)       

In [0]:
# Save comparison chart
val_only = combined[combined["Split"] == "Validation"].copy()
if not val_only.empty and f"F{BETA:.0f} (delayed)" in val_only.columns:
    val_only[f"F{BETA:.0f} (delayed)"] = pd.to_numeric(val_only[f"F{BETA:.0f} (delayed)"], errors="coerce")
    val_only = val_only.dropna(subset=[f"F{BETA:.0f} (delayed)"]).sort_values(f"F{BETA:.0f} (delayed)")

    fig, ax = plt.subplots(figsize=(12, max(6, len(val_only)*0.5)))
    f2c = f"F{BETA:.0f} (delayed)"
    colors = ["#E91E63" if "PyTorch" in str(m) else "#FF9800" if "MLP" in str(m)
              else "#2196F3" for m in val_only["Model"]]
    bars = ax.barh(range(len(val_only)), val_only[f2c].values, color=colors, edgecolor="black", lw=0.5)
    ax.set_yticks(range(len(val_only)))
    ax.set_yticklabels(val_only["Model"].values, fontsize=8)
    ax.set_xlabel("F2 Score (β=2)"); ax.set_title("All Models — Validation F2")
    for bar, val in zip(bars, val_only[f2c].values):
        ax.text(bar.get_width()+0.002, bar.get_y()+bar.get_height()/2,
                f"{val:.4f}", va="center", fontsize=8, fontweight="bold")
    ax.grid(True, axis="x", alpha=0.3); fig.tight_layout()
    fig.savefig(f"{CHART_DIR}/pytorch_vs_pyspark.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {CHART_DIR}/pytorch_vs_pyspark.png")

Saved: /dbfs/FileStore/group_01_02/phase3_charts/pytorch_vs_pyspark.png


## 17. Summary

This notebook trained a **PyTorch MLP** on GPU with modern NN techniques that
PySpark's MLPClassifier does not support:

| Feature | Implementation |
|---|---|
| Activation | ReLU (avoids vanishing gradients) |
| Normalization | BatchNorm1d (stabilizes training) |
| Regularization | Dropout + L2 (weight_decay in Adam) |
| Optimizer | Adam (adaptive learning rate) |
| LR scheduling | ReduceLROnPlateau |
| Early stopping | Validation loss, patience=15 |
| Training curves | Per-epoch train + val loss |

**Experiments:**
- 3 architectures (matching PySpark for direct comparison)
- Dropout ablation (0.0 → 0.5)
- PCA dimensionality reduction experiment
- Threshold tuning (9 thresholds on validation)
- Feature family ablation (8 families)
- Blind test with frozen threshold

**Results:** `dbfs:/student-groups/Group_01_02/phase3_results_summary_mlp_pytorch.csv`
**Charts:** `/dbfs/FileStore/group_01_02/phase3_charts/pytorch_*.png`